# Notebook 02 -- Data Cleaning & Feature Engineering

**Goal:** Transform our raw Backblaze Hard Drive dataset into clean, feature-rich data ready for machine learning.

---

## Why is this the most important step in any ML project?

There's a famous saying in data science: **"Garbage in, garbage out."** Even the most powerful algorithm will produce terrible predictions if you feed it messy, incomplete, or poorly structured data.

Think of it like cooking. Imagine you're a world-class chef, but someone hands you:
- Ingredients with half the labels missing (missing values)
- A mix of items from 50 different cuisines with no recipe (too many unrelated features)
- Raw, unpeeled, unwashed vegetables (raw data)

No matter how skilled you are, the dish will be a mess. **Data cleaning is the prep work** -- washing, peeling, chopping, measuring -- that makes cooking (modeling) possible.

## What is the difference between raw data and processed data?

| Aspect | Raw Data | Processed Data |
|---|---|---|
| **Source** | Straight from the CSV files -- untouched | Cleaned, filtered, and enriched by us |
| **Missing values** | Many columns are full of NaN | No missing values -- every cell has a number |
| **Columns** | 197 columns, most useless | Only the columns that matter for prediction |
| **Features** | Just raw sensor readings | Rolling averages, lag values, rates of change |
| **Ready for ML?** | No | Yes |

## What will we accomplish in this notebook?

**Part A -- Data Cleaning (Steps 1-7):**
1. Load all 92 daily CSV files into one DataFrame
2. Drop columns that are more than 50% empty
3. Fill remaining missing values with 0
4. Filter to the top 3 most common drive models
5. Convert dates to proper datetime format
6. Sort by serial number and date (chronological order)
7. Save the cleaned dataset

**Part B -- Feature Engineering (Steps 8-14):**
8. Create rolling average features (7-day smoothing)
9. Create lag features (what happened 1, 3, 7 days ago)
10. Create rate-of-change features (how fast metrics are changing)
11. Create a drive age feature
12. Drop NaN rows created by rolling/lag operations
13. Save the final processed dataset
14. Summarize all new features

Let's begin!

---

# PART A -- DATA CLEANING

---

## Step 1 -- Load the Dataset and Show the First Few Rows

**What are we doing?**

In Notebook 01, we explored *one day* of data (October 1, 2025) to understand the structure. Now we need to load **all 92 days** (October 1 - December 31, 2025) into a single DataFrame. Why? Because later we'll create features like *"what was this drive's temperature 7 days ago?"* -- and that requires having multiple days of history for each drive.

**Memory optimization -- why we select columns upfront:**

Each daily CSV has 197 columns and ~330,000 rows. Loading all 92 files with all 197 columns would require ~40 GB of RAM. So we'll be smart:

- We'll only load the **metadata columns** we need (date, serial_number, model, capacity_bytes, failure)
- We'll only load the **S.M.A.R.T. raw columns** we identified as useful in Notebook 01
- We skip all `_normalized` columns (we decided raw values are more interpretable)

This drops 197 columns down to ~20, making the load manageable (~4-6 GB).

**How does this connect to the ML pipeline?**

This is the *foundation* -- every future step (cleaning, feature engineering, model training) depends on this DataFrame being loaded correctly.

In [1]:
# ============================================================
# Step 1 -- Load All 92 Daily CSV Files into One DataFrame
# ============================================================
# MEMORY-OPTIMIZED VERSION (for systems with 16 GB RAM)
# Key optimizations:
#   1. Pre-scan to identify top 3 models FIRST
#   2. Filter to top 3 models DURING loading (not after)
#   3. Use dtype specification to avoid float64 allocation
#   4. Garbage collection every 10 files
#   5. Single concat at the end (avoids repeated copies)
#
# This cuts memory from ~6.4 GB down to ~2 GB.

import pandas as pd
import numpy as np
import os
import glob
import gc
import warnings
warnings.filterwarnings('ignore')

# --- Define which columns to load ---
meta_cols = ['date', 'serial_number', 'model', 'capacity_bytes', 'failure']

# S.M.A.R.T. raw columns: 14 with >90% coverage (from Notebook 01)
# plus 3 with <50% coverage so Step 2 has columns to drop
smart_cols = [
    'smart_1_raw',    # Read Error Rate
    'smart_3_raw',    # Spin-Up Time
    'smart_4_raw',    # Start/Stop Count
    'smart_5_raw',    # Reallocated Sectors Count     <- HIGH failure relevance
    'smart_7_raw',    # Seek Error Rate
    'smart_9_raw',    # Power-On Hours
    'smart_10_raw',   # Spin Retry Count
    'smart_12_raw',   # Power Cycle Count
    'smart_187_raw',  # Reported Uncorrectable Errors (low coverage ~34%)
    'smart_188_raw',  # Command Timeout               (low coverage ~34%)
    'smart_190_raw',  # Airflow Temperature            (low coverage ~34%)
    'smart_192_raw',  # Power-Off Retract Count
    'smart_193_raw',  # Load/Unload Cycle Count
    'smart_194_raw',  # Temperature (Celsius)
    'smart_197_raw',  # Current Pending Sector Count  <- HIGH failure relevance
    'smart_198_raw',  # Uncorrectable Sector Count    <- HIGH failure relevance
    'smart_199_raw',  # UltraDMA CRC Error Count
]

use_cols = meta_cols + smart_cols

# --- Dtype specification: avoid float64, use float32 for SMART columns ---
# This halves memory for numeric columns right at parse time.
dtype_spec = {'failure': 'int8'}
for c in smart_cols:
    dtype_spec[c] = 'float32'

# --- Find all CSV files ---
csv_folder = '../data/raw/data_Q4_2025/'
csv_files = sorted(glob.glob(os.path.join(csv_folder, '*.csv')))
print(f'Found {len(csv_files)} CSV files')
print(f'First: {os.path.basename(csv_files[0])}  |  Last: {os.path.basename(csv_files[-1])}')
print()

# ==================================================================
# PASS 1: Quick scan to find the top 3 models (load only 'model')
# ==================================================================
# Instead of loading ALL data and then filtering, we first figure out
# which 3 models are most common. This takes seconds, not minutes.
print('Pass 1: Scanning to identify top 3 drive models...')
from collections import Counter
model_counter = Counter()
for filepath in csv_files:
    models = pd.read_csv(filepath, usecols=['model'], dtype={'model': 'str'})['model']
    model_counter.update(models.value_counts().to_dict())

top_3_models = [m for m, _ in model_counter.most_common(3)]
print(f'Top 3 models: {top_3_models}')
for model, count in model_counter.most_common(3):
    print(f'  {model:<30s}  {count:>10,} rows across all files')
print()

# ==================================================================
# PASS 2: Load all files, filtering to top 3 models immediately
# ==================================================================
# By filtering DURING loading, each file shrinks from ~330K rows to
# ~100-150K rows. This alone cuts memory by 55-70%.
print('Pass 2: Loading data (top 3 models only)...')
BATCH_SIZE = 10
dfs = []

for i, filepath in enumerate(csv_files):
    day_df = pd.read_csv(filepath, usecols=use_cols, dtype=dtype_spec)
    day_df = day_df[day_df['model'].isin(top_3_models)]  # filter immediately
    dfs.append(day_df)
    del day_df

    if (i + 1) % BATCH_SIZE == 0 or (i + 1) == len(csv_files):
        gc.collect()
        total_rows = sum(len(d) for d in dfs)
        print(f'  Loaded {i+1}/{len(csv_files)}: {os.path.basename(filepath)}  '
              f'(collected {total_rows:,} rows so far)')

# --- Single concat at the end (avoids repeated copy-on-concat) ---
df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print()
print(f'Combined DataFrame shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB')
print()
df.head()


Found 92 CSV files
First: 2025-10-01.csv  |  Last: 2025-12-31.csv

Pass 1: Scanning to identify top 3 drive models...
Top 3 models: ['WDC WUH722222ALE6L4', 'TOSHIBA MG08ACA16TA', 'TOSHIBA MG07ACA14TA']
  WDC WUH722222ALE6L4              3,945,245 rows across all files
  TOSHIBA MG08ACA16TA              3,670,933 rows across all files
  TOSHIBA MG07ACA14TA              3,428,534 rows across all files

Pass 2: Loading data (top 3 models only)...
  Loaded 10/92: 2025-10-10.csv  (collected 1,169,271 rows so far)
  Loaded 20/92: 2025-10-20.csv  (collected 2,351,744 rows so far)
  Loaded 30/92: 2025-10-30.csv  (collected 3,546,374 rows so far)
  Loaded 40/92: 2025-11-09.csv  (collected 4,751,610 rows so far)
  Loaded 50/92: 2025-11-19.csv  (collected 5,954,548 rows so far)
  Loaded 60/92: 2025-11-29.csv  (collected 7,167,573 rows so far)
  Loaded 70/92: 2025-12-09.csv  (collected 8,379,545 rows so far)
  Loaded 80/92: 2025-12-19.csv  (collected 9,592,183 rows so far)
  Loaded 90/92: 2025-12-

,date,serial_number,model,capacity_bytes,failure,smart_1_raw,smart_3_raw,smart_4_raw,smart_5_raw,smart_7_raw,...,smart_12_raw,smart_187_raw,smart_188_raw,smart_190_raw,smart_192_raw,smart_193_raw,smart_194_raw,smart_197_raw,smart_198_raw,smart_199_raw
0,2025-10-01,8180A09WFVKG,TOSHIBA MG08ACA16TA,16000900661248,0,0.0,7940.0,5.0,0.0,0.0,...,5.0,NaN,NaN,NaN,3.0,226.0,19.0,0.0,0.0,0.0
1,2025-10-01,2260A086FVKG,TOSHIBA MG08ACA16TA,16000900661248,0,0.0,8006.0,21.0,0.0,0.0,...,20.0,NaN,NaN,NaN,16.0,41.0,36.0,0.0,0.0,0.0
2,2025-10-01,4270A014FVKG,TOSHIBA MG08ACA16TA,16000900661248,0,0.0,7675.0,27.0,0.0,0.0,...,27.0,NaN,NaN,NaN,20.0,46.0,32.0,0.0,0.0,0.0
3,2025-10-01,8130A08YFVKG,TOSHIBA MG08ACA16TA,16000900661248,0,0.0,7882.0,6.0,0.0,0.0,...,6.0,NaN,NaN,NaN,3.0,7.0,40.0,0.0,0.0,0.0
4,2025-10-01,8160A0MEFVKG,TOSHIBA MG08ACA16TA,16000900661248,0,0.0,7970.0,10.0,0.0,0.0,...,10.0,NaN,NaN,NaN,8.0,273.0,29.0,0.0,0.0,1.0


### What just happened?

We used a **two-pass** approach to load data without running out of memory:

- **Pass 1:** Quick scan to identify the top 3 most common drive models (loads only the `model` column -- very fast)
- **Pass 2:** Full load with 3 memory optimizations:
  - `dtype=dtype_spec` forces float32 instead of float64 (halves numeric memory)
  - `.isin(top_3_models)` filter per file (drops ~60-70% of rows immediately)
  - `gc.collect()` after every batch frees unused memory

You should see ~10-14 million rows (only top 3 models) and 22 columns, using ~2 GB instead of 6+ GB.

**Key code concepts:**

| Code | What It Does |
|---|---|
| `glob.glob('*.csv')` | Finds all files matching a pattern |
| `usecols=use_cols` | Tells pandas to only read specific columns -- saves memory |
| `dtype=dtype_spec` | Forces float32/int8 instead of default 64-bit -- halves memory |
| `.isin(top_3_models)` | Filters rows to top 3 models immediately per file |
| `pd.concat(dfs, ignore_index=True)` | Stacks all DataFrames vertically into one big table |
| `gc.collect()` | Forces Python garbage collection to free unused memory |

---


## Step 2 -- Drop Columns That Are More Than 50% Empty

**What are we doing?**

Some S.M.A.R.T. columns have data for only a fraction of drives. If a column is more than 50% NaN, it means **most drives don't report that metric**. Keeping such a column would:

1. **Confuse the model** -- it would have to guess what half the values should be
2. **Add noise** -- the filled-in values might mislead the model
3. **Waste memory** -- carrying useless data slows everything down

Think of it like a survey where half the respondents skipped a question. The answers from the other half might be biased, so the whole column is unreliable.

**What to expect:** Based on Notebook 01, `smart_187_raw` (~34% coverage), `smart_188_raw` (~34%), and `smart_190_raw` (~34%) will likely be dropped.

In [2]:
# ============================================================
# Step 2 -- Drop Columns That Are More Than 50% Empty
# ============================================================

null_pct = df.isnull().mean() * 100

print('=== Null Percentage Per Column ===')
print()
for col in df.columns:
    status = '<-- WILL DROP' if null_pct[col] > 50 else 'keep'
    print(f'  {col:<20s}  {null_pct[col]:6.2f}% null  {status}')

print()
cols_to_drop = null_pct[null_pct > 50].index.tolist()
print(f'Columns to drop ({len(cols_to_drop)}): {cols_to_drop}')

cols_before = df.shape[1]
df.drop(columns=cols_to_drop, inplace=True)
cols_after = df.shape[1]

print(f'Columns: {cols_before} -> {cols_after} (dropped {cols_before - cols_after})')
print(f'Remaining columns: {df.columns.tolist()}')

=== Null Percentage Per Column ===

  date                    0.00% null  keep
  serial_number           0.00% null  keep
  model                   0.00% null  keep
  capacity_bytes          0.00% null  keep
  failure                 0.00% null  keep
  smart_1_raw             0.00% null  keep
  smart_3_raw             0.00% null  keep
  smart_4_raw             0.00% null  keep
  smart_5_raw             0.00% null  keep
  smart_7_raw             0.00% null  keep
  smart_9_raw             0.00% null  keep
  smart_10_raw            0.00% null  keep
  smart_12_raw            0.00% null  keep
  smart_187_raw         100.00% null  <-- WILL DROP
  smart_188_raw         100.00% null  <-- WILL DROP
  smart_190_raw         100.00% null  <-- WILL DROP
  smart_192_raw           0.00% null  keep
  smart_193_raw           0.00% null  keep
  smart_194_raw           0.00% null  keep
  smart_197_raw           0.00% null  keep
  smart_198_raw           0.00% null  keep
  smart_199_raw           0.00% nu

### What just happened?

We calculated the null percentage for every column and dropped those above 50%. You should see:

- **Dropped:** `smart_187_raw`, `smart_188_raw`, `smart_190_raw` -- only ~34% coverage
- **Kept:** All 14 high-coverage columns survived
- **Result:** 22 columns -> 19 columns

**New syntax:** `df.isnull().mean()` creates a True/False grid then calculates the fraction of Trues per column.

---

## Step 3 -- Fill Remaining Missing Values with 0

**What are we doing?**

Even after dropping the worst columns, the surviving ones still have some NaN values (typically 0.5-3% missing). We need to fill these because ML models cannot handle NaN.

**Why fill with 0?**

For S.M.A.R.T. attributes, a missing value almost always means **"no errors were recorded"**. For example:
- `smart_5_raw` (Reallocated Sectors) = NaN -> the drive hasn't reallocated any sectors -> effectively 0
- `smart_197_raw` (Pending Sectors) = NaN -> no sectors are pending -> effectively 0

This is a **domain-specific decision**. In other datasets (e.g., income data), filling with 0 would be wrong. But for error-count sensors, 0 is the correct interpretation.

**WARNING:** If `smart_9_raw` (Power-On Hours) has 0 values after filling, that might indicate brand-new drives rather than missing data.

In [4]:
# ============================================================
# Step 3 -- Fill Remaining Missing Values with 0
# ============================================================

total_nulls_before = df.isnull().sum().sum()
print(f'Total NaN values before filling: {total_nulls_before:,}')
print()

null_counts = df.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]
print('Columns with remaining NaN values:')
for col, count in cols_with_nulls.items():
    pct = count / len(df) * 100
    print(f'  {col:<20s}  {count:>10,} NaN  ({pct:.2f}%)')
print()

df.fillna(0, inplace=True)

total_nulls_after = df.isnull().sum().sum()
print(f'Total NaN values after filling: {total_nulls_after}')
print('All missing values have been filled!' if total_nulls_after == 0 else 'WARNING: Some NaN remain!')

Total NaN values before filling: 0

Columns with remaining NaN values:

Total NaN values after filling: 0
All missing values have been filled!


### What just happened?

We replaced every remaining NaN with 0. The total NaN count is now **0** -- every cell contains a value.

---

### Checkpoint 1 -- Steps 1, 2, 3 Complete

| Step | Action | Result |
|---|---|---|
| 1 | Loaded all 92 daily CSVs | ~30 million rows, 22 columns |
| 2 | Dropped >50% empty columns | 22 -> 19 columns |
| 3 | Filled remaining NaN with 0 | 0 missing values remaining |

**Status:** Complete, gap-free dataset. But it still contains ALL drive models and isn't sorted. Let's fix that next.

---

## Step 4 -- Verify the Top 3 Drive Models

**What are we doing?**

In Step 1, we already filtered to the top 3 most common drive models during loading (to save memory). This step simply **verifies** the distribution and confirms we have the right models.

**Why did we filter so early?**

Loading all ~30 million rows (all models) into memory would require ~6+ GB on a 16 GB system -- causing MemoryError. By filtering during loading, we keep only ~40% of the data, reducing memory to ~2 GB.

**Why these 3 models?**

1. **Consistency** -- the top 3 have the most data, so the model learns the strongest patterns
2. **Simplicity** -- fewer models = fewer confounding variables
3. **Enough data** -- these likely represent >50% of all drives


In [5]:
# ============================================================
# Step 4 -- Verify Top 3 Models (Already Filtered in Step 1)
# ============================================================
# NOTE: We already filtered to top 3 models during loading (Step 1)
# to save memory. This cell just verifies the distribution.

model_counts = df['model'].value_counts()

print(f'Total unique drive models: {len(model_counts)}')
print()
print('Drive model distribution:')
print(model_counts.to_string())
print()

top_3_models = model_counts.head(3).index.tolist()
print(f'Top 3 models: {top_3_models}')
print()
for model in top_3_models:
    count = len(df[df['model'] == model])
    print(f'  {model:<30s}  {count:>10,} rows  ({count/len(df)*100:.1f}%)')
print()
print(f'Total rows: {len(df):,}')


Total unique drive models: 3

Drive model distribution:
model
WDC WUH722222ALE6L4    3945245
TOSHIBA MG08ACA16TA    3670933
TOSHIBA MG07ACA14TA    3428534

Top 3 models: ['WDC WUH722222ALE6L4', 'TOSHIBA MG08ACA16TA', 'TOSHIBA MG07ACA14TA']

  WDC WUH722222ALE6L4              3,945,245 rows  (35.7%)
  TOSHIBA MG08ACA16TA              3,670,933 rows  (33.2%)
  TOSHIBA MG07ACA14TA              3,428,534 rows  (31.0%)

Total rows: 11,044,712


### What just happened?

We verified that the data contains only the top 3 drive models, as expected from our filtered loading in Step 1.

**Key concept:** By filtering early (during loading), we avoided the MemoryError that occurs when loading all ~30 million rows. This is a common optimization called **predicate pushdown** -- applying filters as early as possible in the pipeline.

---


## Step 5 -- Convert the Date Column to Datetime Format

**What are we doing?**

Right now, the `date` column contains strings like `"2025-10-01"`. Python sees this as plain text. By converting to **datetime**, Python understands these are actual dates, enabling:
- Correct sorting (string sorting would put "2025-9-01" after "2025-10-01")
- Date arithmetic (how many days between two readings)
- Time-series features (rolling windows, resampling)

Think of it like the difference between writing "Monday" on a sticky note versus putting it in a calendar app -- the app knows what day comes next.

In [6]:
# ============================================================
# Step 5 -- Convert the Date Column to Datetime
# ============================================================

print(f'Date column type BEFORE: {df["date"].dtype}')
print(f'Sample values: {df["date"].head(3).tolist()}')
print()

df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

print(f'Date column type AFTER: {df["date"].dtype}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Total unique dates: {df["date"].nunique()}')

Date column type BEFORE: object
Sample values: ['2025-10-01', '2025-10-01', '2025-10-01']

Date column type AFTER: datetime64[ns]
Date range: 2025-10-01 00:00:00 to 2025-12-31 00:00:00
Total unique dates: 92


### What just happened?

The dtype changed from `object` (string) to `datetime64[ns]`. Python now knows these are dates. The `format='%Y-%m-%d'` speeds up parsing by telling pandas the exact format.

---

## Step 6 -- Sort by Serial Number and Date

**What are we doing?**

Right now the data is ordered by *file* -- all of October 1's drives first, then October 2's, etc. But for time-series features, we need each drive's readings **in chronological order, grouped together**.

Imagine medical records for 100 patients sorted by *visit date*. To see Patient #42's trend over time, you'd reshuffle so all their records are together, oldest-to-newest. That's what we're doing.

**Why does this matter?**

In Steps 8-10, we'll compute things like "what was this drive's temperature 7 days ago?" These operations look at **previous rows** for each drive. If the data isn't sorted by [serial_number, date], the "previous row" might belong to a different drive.

In [7]:
# ============================================================
# Step 6 -- Sort by serial_number and date
# ============================================================

df.sort_values(by=['serial_number', 'date'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Verify with a sample drive
sample_serial = df['serial_number'].iloc[1000]
sample = df[df['serial_number'] == sample_serial][['date', 'serial_number', 'model', 'smart_194_raw']].head(10)

print(f'Sorted! Showing first 10 days for drive: {sample_serial}')
print()
print(sample.to_string(index=False))
print()
print(f'Unique serial numbers: {df["serial_number"].nunique():,}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Sorted! Showing first 10 days for drive: 1030A034F97G

      date serial_number               model  smart_194_raw
2025-10-01  1030A034F97G TOSHIBA MG07ACA14TA           26.0
2025-10-02  1030A034F97G TOSHIBA MG07ACA14TA           26.0
2025-10-03  1030A034F97G TOSHIBA MG07ACA14TA           27.0
2025-10-04  1030A034F97G TOSHIBA MG07ACA14TA           27.0
2025-10-05  1030A034F97G TOSHIBA MG07ACA14TA           27.0
2025-10-06  1030A034F97G TOSHIBA MG07ACA14TA           26.0
2025-10-07  1030A034F97G TOSHIBA MG07ACA14TA           26.0
2025-10-08  1030A034F97G TOSHIBA MG07ACA14TA           27.0
2025-10-09  1030A034F97G TOSHIBA MG07ACA14TA           26.0
2025-10-10  1030A034F97G TOSHIBA MG07ACA14TA           26.0

Unique serial numbers: 121,885
Shape: 11,044,712 rows x 19 columns


### What just happened?

The DataFrame is now sorted so all readings for each drive are together in chronological order. The sample should show consecutive dates for one drive.

---

### Checkpoint 2 -- Steps 4, 5, 6 Complete

| Step | Action | Result |
|---|---|---|
| 4 | Verified top 3 models | Confirmed correct model filtering |
| 5 | Converted date to datetime | Dates are proper datetime objects |
| 6 | Sorted by serial_number + date | Chronological order per drive |

---


## Step 7 -- Save the Cleaned DataFrame

**What are we doing?**

Before creating new features, let's save a **checkpoint** of the cleaned data. If anything goes wrong in Part B, we can reload from here instead of re-running the slow Step 1.

Think of it like saving your game before a boss fight.

In [8]:
# ============================================================
# Step 7 -- Save the Cleaned DataFrame
# ============================================================

clean_path = '../data/processed/cleaned_drives.csv'
os.makedirs(os.path.dirname(clean_path), exist_ok=True)
df.to_csv(clean_path, index=False)

print('Cleaned dataset saved!')
print(f'   Path: {clean_path}')
print(f'   Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'   File size: {os.path.getsize(clean_path) / (1024**2):.1f} MB')
print()
print('=== Cleaned Dataset Summary ===')
print(f'   Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'   Drive models: {df["model"].unique().tolist()}')
print(f'   Unique drives: {df["serial_number"].nunique():,}')
print(f'   Columns ({df.shape[1]}): {df.columns.tolist()}')
print(f'   Missing values: {df.isnull().sum().sum()}')
print()
print('Failure counts:')
print(df['failure'].value_counts().to_string())

Cleaned dataset saved!
   Path: ../data/processed/cleaned_drives.csv
   Shape: 11,044,712 rows x 19 columns
   File size: 1344.6 MB

=== Cleaned Dataset Summary ===
   Date range: 2025-10-01 to 2025-12-31
   Drive models: ['TOSHIBA MG07ACA14TA', 'TOSHIBA MG08ACA16TA', 'WDC WUH722222ALE6L4']
   Unique drives: 121,885
   Columns (19): ['date', 'serial_number', 'model', 'capacity_bytes', 'failure', 'smart_1_raw', 'smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_7_raw', 'smart_9_raw', 'smart_10_raw', 'smart_12_raw', 'smart_192_raw', 'smart_193_raw', 'smart_194_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw']
   Missing values: 0

Failure counts:
failure
0    11044543
1         169


### What just happened?

We saved the cleaned DataFrame as `cleaned_drives.csv`. The summary confirms correct date range, top 3 models only, zero missing values, and proper sorting.

Notice the **failure counts** -- massive imbalance (very few 1s). We'll handle this in Notebook 03.

---

## Part A Complete -- Data Cleaning Summary

| What | Before | After |
|---|---|---|
| Files | 92 separate CSVs | 1 clean DataFrame |
| Columns | 197 (mostly empty) | 19 (all populated) |
| Models | Hundreds of different drives | Top 3 most common |
| Missing values | Millions of NaN | Zero |
| Date format | Text strings | Proper datetime |
| Sort order | By file (random) | By drive, then date |

The data is now **clean** but not yet *smart*. In Part B, we'll create engineered features that encode patterns over time.

---

# PART B -- FEATURE ENGINEERING

---

## Step 8 -- Create Rolling Average Features (7-Day Window)

**What is a rolling average?**

Imagine tracking your weight daily. Monday: 150, Tuesday: 152, Wednesday: 149... These daily numbers bounce around -- they're **noisy**. But if you average the last 7 days, you get a **smoother trend** showing whether weight is actually going up or down.

A **rolling average** does exactly that. For each row, it averages the **previous 7 rows**. It "rolls" forward day by day:
- Day 7: average of days 1-7
- Day 8: average of days 2-8
- ...and so on

**Why do we need this for failure prediction?**

A single day's reading can spike randomly. But if the **7-day average** is climbing steadily, that's a much stronger warning. Rolling averages help the model see **trends** instead of **noise**.

**Which metrics?**

| Column | Why |
|---|---|
| `smart_5_raw` | Increasing reallocated sectors = drive remapping bad sectors |
| `smart_194_raw` | Sustained high temperature = overheating risk |
| `smart_197_raw` | Rising pending sectors = growing backlog |
| `smart_198_raw` | Rising uncorrectable sectors = data loss risk |
| `smart_199_raw` | Rising CRC errors = connection problems |

**WARNING:** We calculate per drive (grouped by `serial_number`). The first 6 rows per drive will be NaN (not enough history). We fix this in Step 12.

In [9]:
# ============================================================
# Step 8 -- Create Rolling Average Features (7-Day Window)
# ============================================================

key_metrics = ['smart_5_raw', 'smart_194_raw', 'smart_197_raw',
               'smart_198_raw', 'smart_199_raw']

print('Creating 7-day rolling averages...')
print()

# .groupby('serial_number') -- splits data by drive
# .transform(lambda x: ...) -- applies function per group, returns aligned results
# x.rolling(window=7, min_periods=7).mean() -- 7-day rolling average

for col in key_metrics:
    new_col = f'{col}_roll7'
    df[new_col] = (
        df.groupby('serial_number')[col]
        .transform(lambda x: x.rolling(window=7, min_periods=7).mean())
    )
    print(f'  Created: {new_col}')

print()
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print()

# Sample: raw vs rolling for one drive
sample_sn = df['serial_number'].iloc[1000]
sample = df[df['serial_number'] == sample_sn][['date', 'smart_194_raw', 'smart_194_raw_roll7']].head(10)
print(f'Sample -- Drive {sample_sn} (temperature raw vs 7-day rolling):')
print(sample.to_string(index=False))

Creating 7-day rolling averages...

  Created: smart_5_raw_roll7
  Created: smart_194_raw_roll7
  Created: smart_197_raw_roll7
  Created: smart_198_raw_roll7
  Created: smart_199_raw_roll7

Shape: 11,044,712 rows x 24 columns

Sample -- Drive 1030A034F97G (temperature raw vs 7-day rolling):
      date  smart_194_raw  smart_194_raw_roll7
2025-10-01           26.0                  NaN
2025-10-02           26.0                  NaN
2025-10-03           27.0                  NaN
2025-10-04           27.0                  NaN
2025-10-05           27.0                  NaN
2025-10-06           26.0                  NaN
2025-10-07           26.0            26.428571
2025-10-08           27.0            26.571429
2025-10-09           26.0            26.571429
2025-10-10           26.0            26.428571


### What just happened?

We created **5 new columns**. The first 6 rows per drive show NaN (not enough history). From row 7 onward, the rolling average is smoother than raw values.

**Key concept -- `transform()` with `lambda`:** Groups data by drive, applies rolling mean within each group, and returns results aligned with the original DataFrame.

---

## Step 9 -- Create Lag Features (1, 3, and 7 Days Ago)

**What is a lag feature?**

A lag feature answers: *"What was this value N days ago?"*

Think of a **rearview mirror**. You don't just look ahead -- you check what's behind you. Lag features let the model look *backward in time*.

If today's pending sectors = 10 and lag-7 = 2, the model sees: *"Jumped from 2 to 10 in 7 days -- something is wrong."*

**Why 1, 3, and 7 day lags?**

| Lag | What It Captures |
|---|---|
| 1-day | Sudden spikes (yesterday vs today) |
| 3-day | Short-term trends (filters one-day noise) |
| 7-day | Weekly trends (slower deterioration) |

Together they give the model a **multi-scale view** of each drive's history.

**How `shift()` works:**
```
Original:  [10, 12, 15, 18, 20]
shift(1):  [NaN, 10, 12, 15, 18]   <- each value moved down by 1
shift(3):  [NaN, NaN, NaN, 10, 12] <- moved down by 3
```

In [10]:
# ============================================================
# Step 9 -- Create Lag Features (1, 3, and 7 Days Ago)
# ============================================================

lag_periods = [1, 3, 7]

print('Creating lag features...')
print()

for col in key_metrics:
    for lag in lag_periods:
        new_col = f'{col}_lag{lag}'
        df[new_col] = df.groupby('serial_number')[col].shift(lag)
        print(f'  Created: {new_col}')

print()
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print()

sample_sn = df['serial_number'].iloc[1000]
lag_cols = ['date', 'smart_197_raw', 'smart_197_raw_lag1', 'smart_197_raw_lag3', 'smart_197_raw_lag7']
sample = df[df['serial_number'] == sample_sn][lag_cols].head(10)
print(f'Sample -- Drive {sample_sn} (pending sectors: today vs lags):')
print(sample.to_string(index=False))

Creating lag features...

  Created: smart_5_raw_lag1
  Created: smart_5_raw_lag3
  Created: smart_5_raw_lag7
  Created: smart_194_raw_lag1
  Created: smart_194_raw_lag3
  Created: smart_194_raw_lag7
  Created: smart_197_raw_lag1
  Created: smart_197_raw_lag3
  Created: smart_197_raw_lag7
  Created: smart_198_raw_lag1
  Created: smart_198_raw_lag3
  Created: smart_198_raw_lag7
  Created: smart_199_raw_lag1
  Created: smart_199_raw_lag3
  Created: smart_199_raw_lag7

Shape: 11,044,712 rows x 39 columns

Sample -- Drive 1030A034F97G (pending sectors: today vs lags):
      date  smart_197_raw  smart_197_raw_lag1  smart_197_raw_lag3  smart_197_raw_lag7
2025-10-01            0.0                 NaN                 NaN                 NaN
2025-10-02            0.0                 0.0                 NaN                 NaN
2025-10-03            0.0                 0.0                 NaN                 NaN
2025-10-04            0.0                 0.0                 0.0                 NaN

### What just happened?

We created **15 new columns** (3 lags x 5 metrics). Row 1 has all NaN lags (no yesterday). By row 8, all lags have values.

The model can now compare today's value against 1, 3, and 7 days ago to spot trends.

---

## Step 10 -- Create Rate of Change Features

**What is rate of change?**

It answers: *"How much did this value change since yesterday?"*

Think **speedometer** vs **odometer**:
- **Odometer** (raw value): "I've driven 50,000 miles total"
- **Speedometer** (rate of change): "I'm going 70 mph right now"

A drive with 100 reallocated sectors that was at 99 yesterday is much healthier than one that was at 50 yesterday. Same raw value, completely different **rate of change** (+1 vs +50).

**Why is rate of change more informative?**

1. **Direction** -- going up, down, or flat?
2. **Urgency** -- sudden spike vs gradual increase
3. **Normalization** -- removes baseline differences between drives

Real failures are preceded by **accelerating degradation** -- rate of change captures this.

In [11]:
# ============================================================
# Step 10 -- Create Rate of Change Features
# ============================================================

print('Creating rate of change features...')
print()

# Rate of change = today - yesterday (using lag1 from Step 9)
for col in key_metrics:
    new_col = f'{col}_change'
    df[new_col] = df[col] - df[f'{col}_lag1']
    print(f'  Created: {new_col}')

print()
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print()

sample_sn = df['serial_number'].iloc[1000]
change_cols = ['date', 'smart_194_raw', 'smart_194_raw_lag1', 'smart_194_raw_change']
sample = df[df['serial_number'] == sample_sn][change_cols].head(10)
print(f'Sample -- Drive {sample_sn} (temperature: today, yesterday, change):')
print(sample.to_string(index=False))

Creating rate of change features...

  Created: smart_5_raw_change
  Created: smart_194_raw_change
  Created: smart_197_raw_change
  Created: smart_198_raw_change
  Created: smart_199_raw_change

Shape: 11,044,712 rows x 44 columns

Sample -- Drive 1030A034F97G (temperature: today, yesterday, change):
      date  smart_194_raw  smart_194_raw_lag1  smart_194_raw_change
2025-10-01           26.0                 NaN                   NaN
2025-10-02           26.0                26.0                   0.0
2025-10-03           27.0                26.0                   1.0
2025-10-04           27.0                27.0                   0.0
2025-10-05           27.0                27.0                   0.0
2025-10-06           26.0                27.0                  -1.0
2025-10-07           26.0                26.0                   0.0
2025-10-08           27.0                26.0                   1.0
2025-10-09           26.0                27.0                  -1.0
2025-10-10       

### What just happened?

We created **5 new columns**. Positive = metric increased, negative = decreased, zero = no change.

---

### Checkpoint 3 -- Steps 8, 9, 10 Complete

| Step | Action | New Columns |
|---|---|---|
| 8 | Rolling 7-day averages | 5 |
| 9 | Lag features (1, 3, 7 days) | 15 |
| 10 | Rate of change | 5 |

**Total new features so far: 25 columns** transforming raw readings into time patterns.

---

## Step 11 -- Create a Drive Age Feature

**What are we doing?**

`smart_9_raw` contains **Power-On Hours**. We'll convert to **days** to create `drive_age_days`.

**Why does age matter?**

Hard drives follow the **bathtub curve**:

```
Failure  |\                                    /
Rate     | \                                  /
         |  \________________________________/
         |   "Infant"    "Useful Life"    "Wear-out"
         +--------------------------------------> Age
```

1. **Infant mortality:** New drives fail more (manufacturing defects)
2. **Useful life:** Low, stable failure rate
3. **Wear-out:** Old drives fail more (component degradation)

Age is a **contextual feature** -- the same error reading is more concerning in a 5-year-old drive than a 2-year-old one.

In [12]:
# ============================================================
# Step 11 -- Create Drive Age Feature from Power-On Hours
# ============================================================

df['drive_age_days'] = df['smart_9_raw'] / 24

print('Created: drive_age_days (smart_9_raw / 24)')
print()

age = df['drive_age_days'].describe()
print('=== Drive Age Distribution ===')
print(f'  Youngest: {age["min"]:>8.0f} days  ({age["min"]/365:.1f} years)')
print(f'  Median:   {age["50%"]:>8.0f} days  ({age["50%"]/365:.1f} years)')
print(f'  Mean:     {age["mean"]:>8.0f} days  ({age["mean"]/365:.1f} years)')
print(f'  Oldest:   {age["max"]:>8.0f} days  ({age["max"]/365:.1f} years)')
print()
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Created: drive_age_days (smart_9_raw / 24)

=== Drive Age Distribution ===
  Youngest:        0 days  (0.0 years)
  Median:        810 days  (2.2 years)
  Mean:         1005 days  (2.8 years)
  Oldest:       2647 days  (7.3 years)

Shape: 11,044,712 rows x 45 columns


### What just happened?

We converted Power-On Hours to days. Most drives should be 2-4 years old, with some as old as 5+ years. This gives the model crucial bathtub-curve context.

---

## Step 12 -- Drop NaN Rows Created by Rolling and Lag Operations

**Why do NaN rows appear?**

Rolling averages (window=7) and lag features (shift 1, 3, 7) create NaN at the **start of each drive's history**:
- Rolling 7-day: first 6 rows NaN (not enough history)
- Lag-7: first 7 rows NaN (can't look back 7 days)

Think of a movie reviewer who rates based on the last 7 films. For their first 6 reviews, they can't give a "last 7" average.

**Why drop instead of filling with 0?**

- Step 3 NaN meant "drive didn't report this" -- 0 is semantically correct
- Step 12 NaN means "not enough history" -- filling with 0 would **lie to the model**

Better to lose ~7 rows per drive than contaminate features with fake values.

In [13]:
# ============================================================
# Step 12 -- Drop NaN Rows from Rolling/Lag Operations
# ============================================================

rows_before = len(df)
nan_rows = df.isnull().any(axis=1).sum()
print(f'Rows with NaN: {nan_rows:,} ({nan_rows/rows_before*100:.1f}%)')
print()

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

rows_after = len(df)
print(f'Rows: {rows_before:,} -> {rows_after:,} (dropped {rows_before - rows_after:,})')
print(f'Preserved: {rows_after/rows_before*100:.1f}%')
print()

total_nan = df.isnull().sum().sum()
print(f'Remaining NaN: {total_nan}')
print('Dataset is completely clean!' if total_nan == 0 else 'WARNING: NaN remain!')
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Rows with NaN: 853,137 (7.7%)

Rows: 11,044,712 -> 10,191,575 (dropped 853,137)
Preserved: 92.3%

Remaining NaN: 0
Dataset is completely clean!
Final shape: 10,191,575 rows x 45 columns


### What just happened?

We dropped ~7-8% of the data (first 7 days per drive). The remaining ~92% has complete, honest features with zero NaN.

---

## Step 13 -- Save the Final Feature-Engineered Dataset

This is the **final output** of Notebook 02. The file `processed_drives.csv` is what Notebook 03 will load for model training.

**Why `data/processed/` not `data/raw/`?** The raw folder is sacred -- original, untouched data. Processed contains our transformations. Clear lineage: raw -> cleaned -> processed.

In [14]:
# ============================================================
# Step 13 -- Save the Final Processed Dataset
# ============================================================

output_path = '../data/processed/processed_drives.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)

file_mb = os.path.getsize(output_path) / (1024 ** 2)

print('Final processed dataset saved!')
print(f'   Path: {output_path}')
print(f'   Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'   File size: {file_mb:.1f} MB')
print()
print('=== Final Dataset Summary ===')
print(f'   Date range:    {df["date"].min().date()} to {df["date"].max().date()}')
print(f'   Drive models:  {df["model"].unique().tolist()}')
print(f'   Unique drives: {df["serial_number"].nunique():,}')
print(f'   Total rows:    {df.shape[0]:,}')
print(f'   Total columns: {df.shape[1]}')
print(f'   Missing values: {df.isnull().sum().sum()}')

Final processed dataset saved!
   Path: ../data/processed/processed_drives.csv
   Shape: 10,191,575 rows x 45 columns
   File size: 2449.9 MB

=== Final Dataset Summary ===
   Date range:    2025-10-08 to 2025-12-31
   Drive models:  ['TOSHIBA MG07ACA14TA', 'TOSHIBA MG08ACA16TA', 'WDC WUH722222ALE6L4']
   Unique drives: 121,870
   Total rows:    10,191,575
   Total columns: 45
   Missing values: 0


### What just happened?

Saved `processed_drives.csv` -- the exact input for Notebook 03 (Model Training).

---

## Step 14 -- Summary of All New Features Created

Let's create a clear reference of every column and what it means. Critical documentation for future you.

In [ ]:
# ============================================================
# Step 14 -- Print Summary of All Features
# ============================================================

print('=' * 70)
print('  COMPLETE FEATURE REFERENCE -- processed_drives.csv')
print('=' * 70)
print()

print('METADATA COLUMNS (identifiers -- not model inputs):')
print('-' * 70)
for col, desc in {
    'date':           'Date of the reading (datetime)',
    'serial_number':  'Unique ID of the physical hard drive',
    'model':          'Drive model/make (one of top 3)',
    'capacity_bytes': 'Drive storage capacity in bytes',
    'failure':        'TARGET -- 1=failed that day, 0=healthy',
}.items():
    print(f'  {col:<25s}  {desc}')
print()

print('ORIGINAL S.M.A.R.T. COLUMNS (raw sensor readings):')
print('-' * 70)
for col, desc in {
    'smart_1_raw':   'Read Error Rate',
    'smart_3_raw':   'Spin-Up Time',
    'smart_4_raw':   'Start/Stop Count',
    'smart_5_raw':   'Reallocated Sectors [HIGH relevance]',
    'smart_7_raw':   'Seek Error Rate',
    'smart_9_raw':   'Power-On Hours',
    'smart_10_raw':  'Spin Retry Count',
    'smart_12_raw':  'Power Cycle Count',
    'smart_192_raw': 'Power-Off Retract Count',
    'smart_193_raw': 'Load/Unload Cycle Count',
    'smart_194_raw': 'Temperature (Celsius)',
    'smart_197_raw': 'Current Pending Sectors [HIGH relevance]',
    'smart_198_raw': 'Uncorrectable Sectors [HIGH relevance]',
    'smart_199_raw': 'UltraDMA CRC Error Count',
}.items():
    if col in df.columns:
        print(f'  {col:<25s}  {desc}')
print()

print('ENGINEERED FEATURES (created in this notebook):')
print('-' * 70)
print()
print('  Rolling Averages (7-day window -- smooths daily noise):')
for col in key_metrics:
    print(f'    {col}_roll7')
print()
print('  Lag Features (historical lookback):')
for col in key_metrics:
    for lag in [1, 3, 7]:
        print(f'    {col}_lag{lag:<28} Value from {lag} day(s) ago')
print()
print('  Rate of Change (day-over-day difference):')
for col in key_metrics:
    print(f'    {col}_change')
print()
print('  Drive Age:')
print(f'    drive_age_days               Power-on hours / 24')
print()
print('=' * 70)
eng = len([c for c in df.columns if any(x in c for x in ['roll','lag','change']) or c=='drive_age_days'])
smart = len([c for c in df.columns if c.startswith('smart_') and not any(x in c for x in ['roll','lag','change'])])
print(f'TOTAL COLUMNS: {df.shape[1]}')
print(f'  Metadata:            5')
print(f'  Original SMART:      {smart}')
print(f'  Engineered features: {eng}')
print('=' * 70)

---

# Notebook 02 -- Complete Summary

## What We Accomplished

### Part A -- Data Cleaning

| Step | Action | Why |
|---|---|---|
| 1 | Loaded all 92 CSVs | Need multiple days for time-series features |
| 2 | Dropped >50% empty columns | Unreliable columns confuse the model |
| 3 | Filled remaining NaN with 0 | ML models need numbers; 0 is correct for SMART errors |
| 4 | Filtered to top 3 models | Consistent behavior patterns |
| 5 | Converted date to datetime | Required for time-series operations |
| 6 | Sorted by serial_number + date | Chronological order for rolling/lag features |
| 7 | Saved cleaned checkpoint | Avoid re-running slow loading step |

### Part B -- Feature Engineering

We created **26 new features** encoding patterns over time:

| Step | Feature Type | Count | What It Captures |
|---|---|---|---|
| 8 | Rolling 7-day averages | 5 | Smoothed trends |
| 9 | Lag features (1, 3, 7 days) | 15 | Historical context |
| 10 | Rate of change | 5 | Speed and direction |
| 11 | Drive age | 1 | Bathtub curve position |
| 12 | (Dropped NaN rows) | -- | Removed incomplete startup rows |
| 13 | Saved processed_drives.csv | -- | Final output |

## What `processed_drives.csv` Contains

- **Rows:** One hard drive on one day (minus first ~7 startup days per drive)
- **Metadata (5 cols):** date, serial_number, model, capacity_bytes, failure
- **Original SMART (14 cols):** Raw sensor readings surviving the 50% threshold
- **Engineered (26 cols):** Rolling averages, lag values, rates of change, drive age
- **Target variable:** `failure` (0=healthy, 1=failed)
- **No missing values**

## What Notebook 03 (Model Training) Will Do

1. **Load** `processed_drives.csv`
2. **Split** into features (X) and target (y = failure)
3. **Split** into training and testing sets
4. **Handle class imbalance** (failures are extremely rare)
5. **Train** classification models (Random Forest, XGBoost, etc.)
6. **Evaluate** using precision, recall, F1-score (not accuracy -- data is heavily imbalanced)

The features we engineered are the **inputs** to those models. Data scientists say **80% of the work is data preparation** -- and we just finished that 80%.

---

**Notebook 02 -- Data Cleaning & Feature Engineering is complete!**